In [1]:
from datasets import load_dataset

# Load the dataset
dataset = load_dataset("ButterChicken98/plantvillage-image-text-pairs")

# Let's see what's inside
print(dataset)

c:\Users\HP\Documents\project\plantdoc\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/datasets/ButterChicken98/plantvillage-image-text-pairs/resolve/12634582388a2780e284f61885973f5c07e29377/plantvillage-image-text-pairs.py
Retrying in 1s [Retry 1/5].
Using the latest cached version of the dataset since ButterChicken98/plantvillage-image-text-pairs couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'default' at C:\Users\HP\.cache\huggingface\datasets\ButterChicken98___plantvillage-image-text-pairs\default\0.0.0\12634582388a2780e284f61885973f5c07e29377 (last modified on Thu Feb 26 17:09:23 2026).


DatasetDict({
    train: Dataset({
        features: ['image', 'caption', 'captions'],
        num_rows: 20638
    })
})


libraries

In [2]:
import torch
import pandas as pd
import numpy as np

from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import evaluate

CONVERT TO CSV

In [3]:
df = dataset["train"].to_pandas()

# Save as CSV
df.to_csv("dataset.csv", index=False)

print("CSV Saved Successfully ✅")

CSV Saved Successfully ✅


LOAD CSV AND RENAME

In [3]:
df = pd.read_csv("dataset.csv")

print("Original Columns:", df.columns)

# Rename properly
df = df.rename(columns={
    "caption": "label",
    "captions": "text"
})

print("Renamed Columns:", df.columns)
print("Total Rows:", len(df))

df.head()

Original Columns: Index(['image', 'caption', 'captions'], dtype='object')
Renamed Columns: Index(['image', 'label', 'text'], dtype='object')
Total Rows: 20638


,image,label,text
0,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Tomato healthy,['A vibrant green and healthy tomato leaf with...
1,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Tomato Late blight,['A tomato leaf showing dark brown lesions and...
2,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Tomato healthy,['A vibrant green and healthy tomato leaf with...
3,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Tomato mosaic virus,['A tomato leaf with mosaic-like patterns of l...
4,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Pepper bell healthy,['A fresh green bell pepper leaf with a smooth...


ENCODE LABELS

In [4]:
le = LabelEncoder()
df["label"] = le.fit_transform(df["label"])

num_classes = len(le.classes_)
class_names = list(le.classes_)

print("Number of classes:", num_classes)
print("Class names:", class_names)


Number of classes: 15
Class names: ['Pepper bell Bacterial spot', 'Pepper bell healthy', 'Potato Early blight', 'Potato Late blight', 'Potato healthy', 'Tomato Bacterial spot', 'Tomato Early blight', 'Tomato Late blight', 'Tomato Leaf Mold', 'Tomato Septoria leaf spot', 'Tomato Spider mites Two spotted spider mite', 'Tomato Target Spot', 'Tomato YellowLeaf Curl Virus', 'Tomato healthy', 'Tomato mosaic virus']


TRAIN TEST SPLIT

In [5]:
df_train, df_test = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

print("Train Size:", len(df_train))
print("Test Size:", len(df_test))

Train Size: 16510
Test Size: 4128


CONVERT TO HUGGINGFACE DATASET

In [6]:
train_dataset = Dataset.from_pandas(df_train, preserve_index=False)
test_dataset = Dataset.from_pandas(df_test, preserve_index=False)

LOAD TOKENIZER

In [7]:
model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

TOKENIZE DATASET

In [8]:
def tokenize_function(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding=False,
        max_length=128
    )

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

tokenized_train.set_format("torch")
tokenized_test.set_format("torch")

Map: 100%|██████████| 4128/4128 [00:00<00:00, 8416.28 examples/s]


LOAD MODEL

In [9]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_classes
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 2698.45it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


DATA COLLATOR

In [10]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

METRICS

In [11]:
accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return accuracy.compute(predictions=predictions, references=labels)

TRAINING ARGUMENTS

In [12]:
import sys
import importlib

# Reload transformers to use the newly installed version
if 'transformers' in sys.modules:
    importlib.reload(sys.modules['transformers'])

from transformers import TrainingArguments

In [13]:
from transformers import TrainingArguments
training_args = TrainingArguments(
    output_dir="./checkpoints",
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    logging_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none",
    fp16=torch.cuda.is_available()
)

In [29]:
import transformers
print(transformers.__version__)

5.3.0


TRAINER

In [14]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

TRAIN

In [15]:
trainer.train()

c:\Users\HP\Documents\project\plantdoc\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy
1,0.099183,0.000409,1.000000
2,0.000374,0.000116,1.000000
3,0.000144,0.000056,1.000000
4,0.000081,0.000035,1.000000
5,0.000059,0.000029,1.000000


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.24it/s]
c:\Users\HP\Documents\project\plantdoc\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.13it/s]
c:\Users\HP\Documents\project\plantdoc\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.24it/s]
c:\Users\HP\Documents\project\plantdoc\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.66it/s]
c:\Users

TrainOutput(global_step=5160, training_loss=0.019968137033139318, metrics={'train_runtime': 21477.295, 'train_samples_per_second': 3.844, 'train_steps_per_second': 0.24, 'total_flos': 2392518362158080.0, 'train_loss': 0.019968137033139318, 'epoch': 5.0})

SAVE MODEL

In [22]:
save_dir = "./distilbert_plant_model"

trainer.save_model(save_dir)
tokenizer.save_pretrained(save_dir)

print("Model saved successfully ✅")

Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.19s/it]

Model saved successfully ✅


In [16]:
save_dir = "./distilbert_plant_model"

# 1. Manually link 

model.config.id2label = {i: name for i, name in enumerate(class_names)}
model.config.label2id = {name: i for i, name in enumerate(class_names)}

# 2. Now save everything
trainer.save_model(save_dir)
tokenizer.save_pretrained(save_dir)

print("Model saved successfully! ✅")

Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.27s/it]

Model saved successfully! ✅


INFERENCE TEST

In [17]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(save_dir)
tokenizer = AutoTokenizer.from_pretrained(save_dir)

model.to(device)
model.eval()

def predict(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
        pred_id = outputs.logits.argmax(dim=-1).item()
    return class_names[pred_id]

print(predict("Leaves show brown circular spots with yellow halo."))

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 4085.07it/s]


Pepper bell Bacterial spot
